# Évaluation TTS pour SVI téléphonique

Comparaison de **4 moteurs TTS** open source sur des phrases types de SVI en français :

| Moteur | Params | Licence | Notes |
|--------|--------|---------|-------|
| **Kokoro** | 82M | Apache 2.0 | Ultra-rapide, léger |
| **Piper** | ~15M | MIT | CPU, ONNX |
| **Kyutai TTS** | 1.6B | CC-BY 4.0 | Streaming natif, FR natif |
| **CosyVoice 2** | 0.5B | Apache 2.0 | Zero-shot, 9 langues |

Les 4 sont évalués en qualité originale (24kHz) ET en qualité téléphone (G.711 8kHz).

> **Runtime** : sélectionner GPU dans Runtime → Change runtime type → T4 GPU

## 1. Installation

In [ ]:
!pip install -q kokoro soundfile piper-tts pathvalidate moshi sphn
!apt-get install -qq ffmpeg sox libsox-dev > /dev/null 2>&1
print("Installation Kokoro, Piper, Kyutai TTS terminée.")

In [ ]:
# Installation CosyVoice 2 (nécessite git clone)
import os

if not os.path.exists("CosyVoice"):
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git
    %cd CosyVoice
    !pip install -q -r requirements.txt
    !huggingface-cli download FunAudioLLM/CosyVoice2-0.5B --local-dir pretrained_models/CosyVoice2-0.5B
    %cd ..
    print("Installation CosyVoice 2 terminée.")
else:
    print("CosyVoice 2 déjà installé.")

## 2. Chargement des phrases

In [ ]:
import os
from pathlib import Path

# Télécharger phrases.txt depuis le repo
!wget -q https://raw.githubusercontent.com/chris-lmd/eval-tts-svi/master/phrases.txt -O phrases.txt

def charger_phrases(prefixe: str) -> list[tuple[str, str]]:
    phrases = []
    for line in Path("phrases.txt").read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        id_phrase, texte = line.split("|", 1)
        id_phrase = id_phrase.strip()
        texte = texte.strip()
        if id_phrase.startswith(prefixe):
            phrases.append((id_phrase, texte))
    return phrases

phrases_bot = charger_phrases("bot_")
print(f"{len(phrases_bot)} phrases chargées :")
for id_p, texte in phrases_bot:
    print(f"  {id_p}: {texte}")

## 3. G\u00e9n\u00e9ration Kokoro TTS (GPU)

In [ ]:
import time
import soundfile as sf
from kokoro import KPipeline

# Initialiser Kokoro en fran\u00e7ais
kokoro_pipeline = KPipeline(lang_code="f")

# Lister les voix fran\u00e7aises disponibles
print("Pipeline Kokoro initialis\u00e9.")
print("Voix par d\u00e9faut utilis\u00e9e (premi\u00e8re voix fran\u00e7aise disponible).")

In [ ]:
os.makedirs("samples/kokoro", exist_ok=True)

kokoro_times = {}

for id_phrase, texte in phrases_bot:
    output_file = f"samples/kokoro/{id_phrase}.wav"
    start = time.perf_counter()

    generator = kokoro_pipeline(texte, voice="ff_siwis")
    # Concaténer tous les segments
    import numpy as np
    segments = []
    for _, _, audio in generator:
        segments.append(audio)
    full_audio = np.concatenate(segments) if len(segments) > 1 else segments[0]
    sf.write(output_file, full_audio, 24000)

    elapsed_ms = (time.perf_counter() - start) * 1000
    kokoro_times[id_phrase] = elapsed_ms
    print(f"  {id_phrase}: {elapsed_ms:.0f}ms")

print(f"\nMoyenne: {sum(kokoro_times.values())/len(kokoro_times):.0f}ms")

## 4. G\u00e9n\u00e9ration Piper TTS (CPU)

In [ ]:
import subprocess

# T\u00e9l\u00e9charger le mod\u00e8le Piper fran\u00e7ais
os.makedirs("models", exist_ok=True)
if not os.path.exists("models/fr_FR-siwis-medium.onnx"):
    !wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/fr/fr_FR/siwis/medium/fr_FR-siwis-medium.onnx -O models/fr_FR-siwis-medium.onnx
    !wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/fr/fr_FR/siwis/medium/fr_FR-siwis-medium.onnx.json -O models/fr_FR-siwis-medium.onnx.json
    print("Mod\u00e8le Piper t\u00e9l\u00e9charg\u00e9.")
else:
    print("Mod\u00e8le Piper d\u00e9j\u00e0 pr\u00e9sent.")

In [ ]:
os.makedirs("samples/piper", exist_ok=True)

piper_times = {}

for id_phrase, texte in phrases_bot:
    output_file = f"samples/piper/{id_phrase}.wav"
    start = time.perf_counter()

    subprocess.run(
        ["piper", "--model", "models/fr_FR-siwis-medium.onnx",
         "--output_file", output_file],
        input=texte, text=True, check=True,
        capture_output=True,
    )

    elapsed_ms = (time.perf_counter() - start) * 1000
    piper_times[id_phrase] = elapsed_ms
    print(f"  {id_phrase}: {elapsed_ms:.0f}ms")

print(f"\nMoyenne: {sum(piper_times.values())/len(piper_times):.0f}ms")

## 5. Génération Kyutai TTS (GPU)

Modèle `kyutai/tts-1.6b-en_fr` — français natif, streaming, 24kHz.

In [ ]:
from moshi.models.tts import tts_moshika
import sphn
import torch

# Charger le modèle Kyutai TTS (voix féminine française "Moshika")
kyutai_model = tts_moshika(device="cuda")
print(f"Kyutai TTS chargé — sample rate: {kyutai_model.mimi.sample_rate}Hz")

In [ ]:
os.makedirs("samples/kyutai", exist_ok=True)

kyutai_times = {}
sample_rate_kyutai = kyutai_model.mimi.sample_rate  # 24000

for id_phrase, texte in phrases_bot:
    output_file = f"samples/kyutai/{id_phrase}.wav"
    start = time.perf_counter()

    result = kyutai_model.simple_generate(texts=[texte], batch_size=1)
    audio = result[0].cpu()
    sphn.write_wav(output_file, audio, sample_rate_kyutai)

    elapsed_ms = (time.perf_counter() - start) * 1000
    kyutai_times[id_phrase] = elapsed_ms
    print(f"  {id_phrase}: {elapsed_ms:.0f}ms")

print(f"\nMoyenne: {sum(kyutai_times.values())/len(kyutai_times):.0f}ms")

# Libérer la VRAM
del kyutai_model
torch.cuda.empty_cache()

## 6. Génération CosyVoice 2 (GPU)

Modèle `FunAudioLLM/CosyVoice2-0.5B` — 9 langues dont FR, zero-shot, 24kHz.

> CosyVoice utilise le **zero-shot** : il clone une voix à partir d'un échantillon audio de référence. On utilise ici un des échantillons Kokoro comme référence pour avoir une voix féminine française comparable.

In [ ]:
import sys
sys.path.insert(0, "CosyVoice")
sys.path.insert(0, "CosyVoice/third_party/Matcha-TTS")
from cosyvoice.cli.cosyvoice import AutoModel
import torchaudio

cosyvoice = AutoModel(model_dir="CosyVoice/pretrained_models/CosyVoice2-0.5B")
print(f"CosyVoice 2 chargé — sample rate: {cosyvoice.sample_rate}Hz")

# Utiliser le premier échantillon Kokoro comme voix de référence
voix_reference = "samples/kokoro/bot_01.wav"
# Texte de référence (doit correspondre au contenu audio)
texte_reference = dict(phrases_bot)["bot_01"]
print(f"Voix de référence: {voix_reference}")

In [ ]:
os.makedirs("samples/cosyvoice", exist_ok=True)

cosyvoice_times = {}

for id_phrase, texte in phrases_bot:
    output_file = f"samples/cosyvoice/{id_phrase}.wav"
    start = time.perf_counter()

    # Zero-shot : cloner la voix de référence
    for i, result in enumerate(cosyvoice.inference_zero_shot(
        texte, texte_reference, voix_reference
    )):
        torchaudio.save(output_file, result["tts_speech"], cosyvoice.sample_rate)

    elapsed_ms = (time.perf_counter() - start) * 1000
    cosyvoice_times[id_phrase] = elapsed_ms
    print(f"  {id_phrase}: {elapsed_ms:.0f}ms")

print(f"\nMoyenne: {sum(cosyvoice_times.values())/len(cosyvoice_times):.0f}ms")

# Libérer la VRAM
del cosyvoice
torch.cuda.empty_cache()

## 7. Conversion qualité téléphone (G.711 8kHz)

In [ ]:
for engine in ["kokoro", "piper", "kyutai", "cosyvoice"]:
    tel_dir = f"samples/telephonie/{engine}"
    os.makedirs(tel_dir, exist_ok=True)
    for wav_file in sorted(Path(f"samples/{engine}").glob("*.wav")):
        output_file = f"{tel_dir}/{wav_file.name}"
        subprocess.run(
            ["ffmpeg", "-i", str(wav_file),
             "-ar", "8000", "-ac", "1", "-acodec", "pcm_mulaw",
             output_file, "-y"],
            check=True, capture_output=True,
        )
    print(f"{engine}: {len(list(Path(tel_dir).glob('*.wav')))} fichiers convertis en qualité téléphone")

## 8. Écoute comparative

Écoutez chaque phrase dans les 8 versions (4 moteurs x 2 qualités) :
1. Kokoro — originale / téléphone
2. Piper — originale / téléphone
3. Kyutai TTS — originale / téléphone
4. CosyVoice 2 — originale / téléphone

In [ ]:
from IPython.display import display, Audio, HTML

moteurs = [
    ("kokoro", "Kokoro", "24kHz"),
    ("piper", "Piper", "22kHz"),
    ("kyutai", "Kyutai TTS", "24kHz"),
    ("cosyvoice", "CosyVoice 2", "24kHz"),
]

for id_phrase, texte in phrases_bot:
    display(HTML(f"<h3>{id_phrase}</h3>"))
    display(HTML(f"<p><em>{texte}</em></p>"))

    for dossier, nom, sr in moteurs:
        display(HTML(f"<b>{nom} (originale {sr})</b>"))
        display(Audio(f"samples/{dossier}/{id_phrase}.wav"))

        display(HTML(f"<b>{nom} (téléphone 8kHz)</b>"))
        display(Audio(f"samples/telephonie/{dossier}/{id_phrase}.wav"))

    display(HTML("<hr>"))

## 9. Comparaison latences

In [ ]:
display(HTML("<h3>Latences de génération (ms)</h3>"))

tous_les_temps = {
    "Kokoro": kokoro_times,
    "Piper": piper_times,
    "Kyutai": kyutai_times,
    "CosyVoice": cosyvoice_times,
}

header = f"{'Phrase':<25} {'Kokoro':>10} {'Piper':>10} {'Kyutai':>10} {'CosyVoice':>12}"
print(header)
print("-" * len(header))
for id_phrase, _ in phrases_bot:
    vals = [t.get(id_phrase, 0) for t in tous_les_temps.values()]
    min_val = min(vals)
    markers = ["*" if v == min_val else " " for v in vals]
    print(f"{id_phrase:<25} {vals[0]:>8.0f}ms {vals[1]:>8.0f}ms {vals[2]:>8.0f}ms {vals[3]:>10.0f}ms")

print("-" * len(header))
moyennes = [sum(t.values()) / len(t) for t in tous_les_temps.values()]
print(f"{'MOYENNE':<25} {moyennes[0]:>8.0f}ms {moyennes[1]:>8.0f}ms {moyennes[2]:>8.0f}ms {moyennes[3]:>10.0f}ms")

## 10. Grille d'évaluation

Remplissez cette grille après écoute :

| Critère | Kokoro | Piper | Kyutai TTS | CosyVoice 2 | Notes |
|---------|--------|-------|------------|-------------|-------|
| Naturel de la voix | /5 | /5 | /5 | /5 | |
| Prosodie / rythme | /5 | /5 | /5 | /5 | |
| Clarté en qualité téléphone | /5 | /5 | /5 | /5 | **Le plus important** |
| Prononciation "Ma-if" | /5 | /5 | /5 | /5 | |
| Latence | ms | ms | ms | ms | |
| **Score total** | /25 | /25 | /25 | /25 | |

## 11. Télécharger les WAV

In [ ]:
# Cr\u00e9er une archive ZIP avec tous les \u00e9chantillons
!zip -r echantillons_tts.zip samples/

from google.colab import files
files.download("echantillons_tts.zip")
print("T\u00e9l\u00e9chargement lanc\u00e9.")